In [7]:
import os
from typing import Optional
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

# Load environment variables from .env file
load_dotenv()

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., OPENAI_API_KEY)
try:
    llm: Optional[ChatOpenAI] = ChatOpenAI(
        model=os.getenv("DEEPSEEK_MODEL_ID"),
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        temperature=0.7
    )
    if llm:
        print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None


# --- Define Independent Chains ---
# These three chains represent distinct tasks that can be executed in parallel.

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Parallel + Synthesis Chain ---

# 1. Define the block of tasks to run in parallel. The results of these,
#    along with the original topic, will be fed into the next step.
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),  # Pass the original topic through
    }
)

# 2. Define the final synthesis prompt which will combine the parallel results.
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])

# 3. Build the synthesis chain, then retain both the parallel results and
#    the final synthesized response in the same result dictionary.
synthesis_chain = synthesis_prompt | llm | StrOutputParser()
full_parallel_chain = (
    map_chain
    | RunnablePassthrough.assign(final=synthesis_chain)
)


# --- Run the Chain ---
async def run_parallel_example(topic: str) -> None:
    """
    Asynchronously invokes the parallel processing chain with a specific topic
    and prints the synthesized result.

    Args:
        topic: The input topic to be processed by the LangChain chains.
    """
    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    try:
        # The input to `ainvoke` is the single 'topic' string, which is
        # then passed to each runnable in the `map_chain`.
        result = await full_parallel_chain.ainvoke(topic)

        print("\n--- Summary Chain ---")
        print(result["summary"])

        print("\n--- Questions Chain ---")
        print(result["questions"])

        print("\n--- Key Terms Chain ---")
        print(result["key_terms"])

        print("\n--- Final Synthesis Chain ---")
        print(result["final"])
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

if __name__ == "__main__":
    test_topic = "AI Agent技术发展历程"
    # Jupyter supports top-level await and already owns the event loop.
    await run_parallel_example(test_topic)

Language model initialized: deepseek-v4-flash

--- Running Parallel LangChain Example for Topic: 'AI Agent技术发展历程' ---

--- Summary Chain ---
AI Agent技术发展大致经历三个阶段：

1. **规则与符号驱动阶段（1950s–1990s）**：以专家系统、BDI模型、反应式Agent为代表，依赖人工规则和逻辑推理，能力有限、场景封闭。
2. **学习与多智能体阶段（1990s–2010s）**：强化学习、多智能体系统、机器人控制发展，Agent开始具备环境感知、决策与协作能力，但泛化性不足。
3. **大模型驱动阶段（2020s至今）**：以LLM为核心，结合规划、记忆、工具调用、RAG和多Agent协作，出现ReAct、AutoGPT、LangChain等范式，Agent从“规则执行”走向“自主理解与任务执行”。

总体趋势：**从符号规则到数据学习，再到以大模型为大脑的通用化、工具化、多智能体化自主Agent**。

--- Questions Chain ---
1. 从符号主义专家系统到基于大语言模型的智能体，AI Agent 的“自主性”边界是如何被一次次重新定义的？
2. 在 AI Agent 技术发展历程中，真正推动范式转变的关键节点是模型能力突破、工具调用机制，还是人机交互方式的改变？
3. 回顾 AI Agent 的发展史，哪些早期失败路径或被放弃的架构，反而为今天的记忆机制、多智能体协作和规划能力埋下了伏笔？

--- Key Terms Chain ---
符号主义, 专家系统, 强化学习, 多智能体系统, 大语言模型, 工具调用, 规划与记忆, 自主智能体, 多模态智能体

--- Final Synthesis Chain ---
综合来看，AI Agent 的发展不是单点技术替代，而是“认知核心、规划与记忆、工具与行动、多智能体协作、人机监督”不断重新组合的过程。其主线是：**Agent 的自主性从“在封闭规则内选择和执行”，扩展到“在训练环境中学习策略”，再扩展到“在开放自然语言任务中理解目标、规划、调用工具并协作执行”**；但自主性始终是受权限、验证、安全与责任边界约束的“委托自主”，而非完全